# 🧪 Prompt A/B Testing with LangChain, OpenAI & LangSmith

**Objective:** evaluate the impact of different prompt variants on AI-generated responses by running A/B tests with LangChain and OpenAI, and identify the most effective prompt style for improved output quality.

### In a nutshell
We take one fixed paragraph and ask the model to summarize it **four different ways** — four phrasings of the same instruction. Every call is traced to LangSmith, and the results are laid out in a comparison table. By holding the input constant and varying only the prompt, we can see — and *measure* — which prompt style produces the best summary.

### Why A/B test prompts?
The wording of a prompt changes the model's output, sometimes a lot. Rather than guessing which phrasing works best, an A/B test makes the comparison systematic: same input, different instructions, side-by-side results. LangSmith then adds the numbers behind each response — tokens, latency, and cost — so "which prompt is better?" becomes a question you can answer with data.

### The pipeline
```
   text_to_summarize  (one fixed paragraph)
              │
              ▼
   ┌──────────────────────────────┐
   │  prompt_variants (4 phrasings)│   Variant 1 · 2 · 3 · 4
   └──────────────────────────────┘
              │  for each variant:  prefix + text
              ▼
   ┌──────────────────────────────┐        ┌────────────────────────┐
   │          ChatOpenAI          │ ─────▶ │    OpenAI Chat API     │
   │          llm.invoke()        │ ◀───── │    (e.g. gpt-5-mini)   │
   └──────────────────────────────┘        └────────────────────────┘
              │  response.content  (one per variant)
              ▼
   ┌──────────────────────────────┐
   │   comparison table (tabulate)│   Variant | Prompt | AI Response
   └──────────────────────────────┘

   ══ all 4 calls are traced automatically ══▶  📊 LangSmith project "A-B Testing Demo"
```

Each loop iteration is one arm of the A/B test — and one run you can inspect in LangSmith.

## Step 1 — Install the dependencies

Run once per environment. `langchain-openai` gives us `ChatOpenAI`, `python-dotenv` loads secrets from a `.env` file, and `tabulate` renders the side-by-side comparison table at the end. Restart the kernel afterwards so the new packages load cleanly.

In [ ]:
%pip install langchain langchain-openai langchain-community python-dotenv
%pip install tabulate

## Step 2 — Imports and load secrets

`load_dotenv()` reads your `.env` file into environment variables, so keys never live in the notebook itself. For this notebook your `.env` should define `OPENAI_API_KEY`, `LANGSMITH_API_KEY`, and `OPENAI_MODEL_NAME` (e.g. `gpt-5-mini`).

In [2]:
import os
from dotenv import load_dotenv
from tabulate import tabulate
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv()  # Load environment variables from .env file

True

## Step 3 — Create the model and turn on LangSmith tracing

Two things happen here:
- **`ChatOpenAI(..., temperature=1)`** creates the model client. Note `temperature=1` makes the output *creative and non-deterministic* — re-running gives slightly different wording each time. (Set `temperature=0` if you want variants to differ only because of the prompt, not randomness.)
- **`LANGSMITH_TRACING="true"`** switches tracing on and **`LANGSMITH_PROJECT`** names the folder (`A-B Testing Demo`) your runs land in. Because tracing is global, *every* `llm.invoke()` in this notebook is captured automatically — no `@traceable` decorator needed. The sanity check confirms tracing is on without printing any secret.

In [3]:
# ---------- Turn on LangSmith tracing (do this before any llm calls) ----------
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "A-B Testing Demo"

# ---------- OpenAI Chat Model ----------
llm = ChatOpenAI(
    model_name=os.environ["OPENAI_MODEL_NAME"],
    temperature=1
)

# ---------- Sanity check (no secrets printed) ----------
print(
    "Tracing on?:", os.getenv("LANGSMITH_TRACING") == "true",
    "| Project:", os.getenv("LANGSMITH_PROJECT"),
    "| Has LangSmith key?:", bool(os.getenv("LANGSMITH_API_KEY"))
)

Tracing on?: True | Project: A-B Testing Demo | Has LangSmith key?: True


## Step 4 — Define the input and the prompt variants

This is the heart of the A/B test. `text_to_summarize` holds one short paragraph about artificial intelligence — the *fixed input* every variant sees. `prompt_variants` is a list of **four different phrasings** of the same instruction ("summarize this"), from a formal one-sentence request down to a casual `TL;DR:`. Keeping the input constant while changing only the instruction is what lets us isolate and compare how prompt wording influences the model's response.

In [4]:
# ---------- Text to Summarize ----------
text_to_summarize = (
    "Artificial Intelligence is revolutionizing industries by enabling machines "
    "to learn from data, adapt to new inputs, and perform tasks that normally "
    "require human intelligence."
)
# ---------- Prompt Variants ----------
prompt_variants = [
    "Summarize the following in one sentence:\n",
    "Write a short summary of this paragraph:\n",
    "Condense this into a concise statement:\n",
    "TL;DR:\n"
]

## Step 5 — Run the A/B test and compare

This loop is the experiment. For each prompt variant we combine it with the fixed input (`prefix + text_to_summarize`), call `llm.invoke()` to get the model's response, and collect the results. The `tabulate` library then prints a comparison table showing the **variant, the prompt text, and the AI-generated summary** side by side — making it easy to eyeball how different wordings affect the output.

Each `llm.invoke()` also fires its own trace to LangSmith, so after this runs you'll have **four runs** in the project ready to compare quantitatively (tokens, latency, cost).

In [5]:
# ---------- Run Variants ----------
results = []
for idx, prefix in enumerate(prompt_variants):
    full_prompt = prefix + text_to_summarize
    response = llm.invoke(
        [HumanMessage(content=full_prompt)]
    )
    results.append([
        f"Variant {idx+1}",
        prefix.strip(),
        response.content.strip()
    ])
# ---------- Display Results ----------
print(tabulate(results, headers=["Variant", "Prompt", "AI Response"]))

Variant    Prompt                                    AI Response
---------  ----------------------------------------  ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Variant 1  Summarize the following in one sentence:  Artificial Intelligence is transforming industries by allowing machines to learn from data, adjust to new information, and execute tasks typically requiring human intelligence.
Variant 2  Write a short summary of this paragraph:  Artificial Intelligence is transforming industries by allowing machines to learn from data, adapt to new information, and execute tasks that typically require human intelligence.
Variant 3  Condense this into a concise statement:   Artificial Intelligence is transforming industries by enabling machines to learn from data and perform tasks typically requiring human intelligence.
Variant 4  TL;DR:                      

## 📊 What to observe in LangSmith

Open [smith.langchain.com](https://smith.langchain.com) → **Projects → `A-B Testing Demo`**. Because tracing is on, the loop above logged **one run per prompt variant — four runs in total**. This is where the A/B test becomes measurable rather than just visual. Look for:

- **Four separate `ChatOpenAI` runs**, one per variant. Sort by time; the newest four are from your latest execution.
- **Inputs.** Open each run and confirm the *only* thing that changed is the instruction prefix — the paragraph is identical. That's what makes it a clean A/B test.
- **Outputs.** The summary each prompt produced — compare against the table printed in the notebook.
- **Token usage.** Prompt vs. completion vs. total tokens per variant. A `TL;DR:` prompt is far shorter than "Write a short summary of this paragraph:", so its prompt-token count should be lower — a real, quantitative difference between variants.
- **Latency.** How long each variant took; useful when a prompt style consistently runs faster or slower.
- **Cost.** Estimated $ per run — multiply across a real workload to see which prompt is cheapest at scale.
- **Model & parameters.** Confirm `temperature=1` is recorded. Re-run the notebook and you'll get a *new* set of four runs with slightly different outputs — proof that temperature adds randomness on top of the prompt effect.

**Try this:** run the notebook 2–3 times, then use LangSmith's table view to compare token counts and latency across all variants at once. Which prompt style gives you the summary quality you want for the fewest tokens? That trade-off — quality vs. cost vs. speed — is exactly what A/B testing prompts is meant to reveal.

### Key takeaways
| Concept | Takeaway |
|---|---|
| A/B testing prompts | Hold the input fixed, vary only the instruction, and compare outputs to find the most effective phrasing. |
| Automatic tracing | With `LANGSMITH_TRACING="true"`, every `llm.invoke()` is captured — no decorator required. |
| Quantitative comparison | LangSmith turns "which looks better?" into measurable tokens, latency, and cost per variant. |
| `temperature` | At `temperature=1` outputs vary run-to-run; drop to `0` to attribute differences to the prompt alone. |